In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import mannwhitneyu
from statsmodels.stats.proportion import proportions_ztest

df = pd.read_csv("/content/cookie_cats.csv")
df.head()

,userid,version,sum_gamerounds,retention_1,retention_7
0,116,gate_30,3,False,False
1,337,gate_30,38,True,False
2,377,gate_40,165,True,False
3,483,gate_40,1,False,False
4,488,gate_40,179,True,True


In [ ]:
df.shape
df.info()
df.isnull().sum()
df.duplicated("userid").sum()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 90189 entries, 0 to 90188
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   userid          90189 non-null  int64 
 1   version         90189 non-null  object
 2   sum_gamerounds  90189 non-null  int64 
 3   retention_1     90189 non-null  bool  
 4   retention_7     90189 non-null  bool  
dtypes: bool(2), int64(2), object(1)
memory usage: 2.2+ MB


np.int64(0)

In [ ]:
df["sum_gamerounds"].describe()

,sum_gamerounds
count,90189.000000
mean,51.872457
std,195.050858
min,0.000000
25%,5.000000
50%,16.000000
75%,51.000000
max,49854.000000


In [ ]:
version_counts = df["version"].value_counts()
version_counts

,count
version,
gate_40,45489
gate_30,44700


In [ ]:
summary = df.groupby("version").agg(
    players=("userid", "count"),
    avg_rounds=("sum_gamerounds", "mean"),
    median_rounds=("sum_gamerounds", "median"),
    d1_retention=("retention_1", "mean"),
    d7_retention=("retention_7", "mean")
).reset_index()

summary

,version,players,avg_rounds,median_rounds,d1_retention,d7_retention
0,gate_30,44700,52.456264,17.0,0.448188,0.190201
1,gate_40,45489,51.298776,16.0,0.442283,0.182000


In [ ]:
df["sum_gamerounds"].sort_values(ascending=False).head(10)

,sum_gamerounds
57702,49854
7912,2961
29417,2640
43671,2438
48188,2294
46344,2251
87007,2156
36933,2124
88328,2063
6536,2015


In [ ]:
df_engagement = df[df["sum_gamerounds"] < 10000]
df_engagement.shape

(90188, 5)

In [ ]:
engagement_summary = df_engagement.groupby("version").agg(
    players=("userid", "count"),
    avg_rounds=("sum_gamerounds", "mean"),
    median_rounds=("sum_gamerounds", "median")
).reset_index()

engagement_summary

,version,players,avg_rounds,median_rounds
0,gate_30,44699,51.342111,17.0
1,gate_40,45489,51.298776,16.0


In [ ]:
counts = df.groupby("version")["retention_1"].sum()
nobs = df.groupby("version")["retention_1"].count()

z_stat, p_value = proportions_ztest(counts, nobs)
retention_1_summary = pd.DataFrame({
    "retained_users": counts,
    "total_users": nobs,
    "retention_rate": (counts / nobs * 100).round(2)
})

retention_1_summary

,retained_users,total_users,retention_rate
version,,,
gate_30,20034,44700,44.82
gate_40,20119,45489,44.23


In [ ]:
counts = df.groupby("version")["retention_7"].sum()
nobs = df.groupby("version")["retention_7"].count()

z_stat, p_value = proportions_ztest(counts, nobs)

counts_7 = df.groupby("version")["retention_7"].sum()
nobs_7 = df.groupby("version")["retention_7"].count()

retention_7_summary = pd.DataFrame({
    "retained_users": counts_7,
    "total_users": nobs_7,
    "retention_rate": (counts_7 / nobs_7 * 100).round(2)
})

retention_7_summary

,retained_users,total_users,retention_rate
version,,,
gate_30,8502,44700,19.02
gate_40,8279,45489,18.20


In [ ]:
retention_summary = df.groupby("version").agg(
    players=("userid", "count"),
    d1_retained=("retention_1", "sum"),
    d7_retained=("retention_7", "sum"),
    d1_retention_rate=("retention_1", "mean"),
    d7_retention_rate=("retention_7", "mean")
).reset_index()

retention_summary["d1_retention_rate"] = (retention_summary["d1_retention_rate"] * 100).round(2)
retention_summary["d7_retention_rate"] = (retention_summary["d7_retention_rate"] * 100).round(2)

retention_summary

,version,players,d1_retained,d7_retained,d1_retention_rate,d7_retention_rate
0,gate_30,44700,20034,8502,44.82,19.02
1,gate_40,45489,20119,8279,44.23,18.20


In [ ]:
gate30_rounds = df_engagement[df_engagement["version"] == "gate_30"]["sum_gamerounds"]
gate40_rounds = df_engagement[df_engagement["version"] == "gate_40"]["sum_gamerounds"]

u_stat, p_value = mannwhitneyu(gate30_rounds, gate40_rounds, alternative="two-sided")

print("U-stat:", u_stat)
print("p-value:", p_value)

U-stat: 1024285761.5
p-value: 0.05089155279145376


In [ ]:
df.to_csv("cookie_cats_clean.csv", index=False)
summary.to_csv("ab_test_summary.csv", index=False)

In [ ]:
ab_test_summary = pd.DataFrame({
    "Metric": ["Day-1 Retention", "Day-7 Retention", "Median Game Rounds"],
    "Gate 30": ["44.82%", "19.02%", "17"],
    "Gate 40": ["44.23%", "18.20%", "16"],
    "Difference": ["+0.59 pp", "+0.82 pp", "+1"],
    "p-value": ["0.074", "0.0016", "0.051"],
    "Result": ["Not significant", "Significant", "Borderline"]
})

ab_test_summary.to_csv("ab_test_summary.csv", index=False)